# <div align="center">🌟 Madar AI - DeepSeek-R1-Distill-1.5B Bilingual LoRA Studio</div>
### <div align="center">Reasoning & Chain-of-Thought Ring Specialist (Arabic & English)</div>

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/azzeddine2017/Madar/blob/main/notebooks/Madar_LoRA_DeepSeek_R1_1.5B.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Madar-blue.svg)](https://github.com/azzeddine2017/Madar)
[![RingTensor](https://img.shields.io/badge/Engine-Native%20Madar%20C%20Kernel-green.svg)]()
[![LoRA](https://img.shields.io/badge/Fine--Tuning-28--Layer%20LoRA-purple.svg)]()

---

This studio trains and fine-tunes **DeepSeek-R1-Distill-Qwen-1.5B** specifically for the **Ring programming language** using Madar's native engine.

### 🎯 Key Engineering Highlights:
- **Chain-of-Thought Reasoning**: Preserves the `<think>...</think>` internal reasoning capabilities of DeepSeek-R1 while directing output toward idiomatic Ring code.
- **Zero Python ML Overhead**: 100% native execution in Ring and Madar C Core (`language/src/ring_inference.c`).
- **True Multi-Layer LoRA**: Adapters across **all 28 Transformer layers** ($A_Q, B_Q, A_{Out}, B_{Out}$ totaling 56 attention tensors).
- **Exact Causal Next-Token Cross-Entropy**: Mathematical loss derivation with AdamW optimization and learning rate cosine schedule.
- **High-Precision Decimal Reporting**: 6-decimal loss logging (`decimals(6)`) for granular convergence verification.
- **Zero-Dependency GGUF Fusion**: Direct export of fused standalone `.gguf` file ready for deployment in Madar, Ollama, and LM Studio.

## ⚙️ Step 1: Hardware & Environment Verification
Verify NVIDIA Tesla GPU (T4 / V100 / A100) and CPU SIMD acceleration.

In [ ]:
# 1. Check NVIDIA GPU status
!nvidia-smi

# 2. Check CPU AVX2 & FMA instruction support
!lscpu | grep -E "Model name|avx2|fma"

## 🔨 Step 2: Clone Repository & Build Madar Runtime Engine
Synchronizes the codebase and builds the Core Runtime (`bin/ring`, `lib/libring.so`, and extensions) via GCC with OpenCL GPU support.

In [ ]:
import os, sys

# 1. Clone repository or pull latest updates
if not os.path.exists("/content/Madar/language"):
    !git clone https://github.com/azzeddine2017/Madar.git /content/Madar
    %cd /content/Madar
else:
    %cd /content/Madar
    !git fetch origin main
    !git reset --hard origin/main

# 2. Install essential dependencies for OpenCL & GCC
!apt-get update -qq && apt-get install -y -qq build-essential g++ ocl-icd-opencl-dev opencl-headers curl libcurl4-openssl-dev

# 3. Execute the Linux build pipeline for Core Runtime & Extensions
!bash tools/build_linux.sh

# 4. Align Shared Library Paths
!mkdir -p /content/Madar/lib /content/Madar/bin /content/Madar/extensions/ringtensor /content/Madar/extensions/ringcjson
!cp -f /content/Madar/lib/libring_tensor.so /content/Madar/bin/libring_tensor.so 2>/dev/null || true
!cp -f /content/Madar/lib/libring_tensor.so /content/Madar/extensions/ringtensor/libring_tensor.so 2>/dev/null || true
!cp -f /content/Madar/lib/libring_cjson.so /content/Madar/bin/libring_cjson.so 2>/dev/null || true
!cp -f /content/Madar/lib/libring_cjson.so /content/Madar/extensions/ringcjson/libring_cjson.so 2>/dev/null || true

os.environ["LD_LIBRARY_PATH"] = "/content/Madar/lib:/content/Madar/bin:/content/Madar/extensions/ringtensor:/content/Madar/extensions/ringcjson:" + os.environ.get("LD_LIBRARY_PATH", "")

# 5. Verify Core Binary
!bin/ring -v

## 📥 Step 3: Download DeepSeek-R1-Distill-1.5B Base Model
Download the official base quantized GGUF model (`DeepSeek-R1-Distill-Qwen-1.5B-Q4_0.gguf`).

In [ ]:
!mkdir -p models
# Download DeepSeek-R1-Distill-Qwen-1.5B GGUF
!wget -c -q --show-progress -O models/deepseek-r1-distill-qwen-1.5b-q4_0.gguf \
    https://huggingface.co/bartowski/DeepSeek-R1-Distill-Qwen-1.5B-GGUF/resolve/main/DeepSeek-R1-Distill-Qwen-1.5B-Q4_0.gguf

!ls -lh models/deepseek-r1-distill-qwen-1.5b-q4_0.gguf

## 📚 Step 4: Verify Bilingual Instruction Training Dataset
Ingest and inspect the **1,460 bilingual (Arabic & English) Ring instruction pairs** in `data/ring_instruct_bilingual.jsonl`.

In [ ]:
# Verify dataset line count and samples
!wc -l data/ring_instruct_bilingual.jsonl
!head -n 2 data/ring_instruct_bilingual.jsonl

## 🪄 Step 5: Register Interactive `%%madar` Cell Magic
Enables real-time streaming execution of Ring / Madar scripts inside Google Colab.

In [ ]:
from IPython.core.magic import register_cell_magic
import subprocess, tempfile, os, sys

MADAR_BIN = os.path.abspath("bin/ring")

@register_cell_magic
def madar(line, cell):
    """IPython Cell Magic to execute Madar code with real-time streamed output."""
    with tempfile.NamedTemporaryFile("w", suffix=".ring", delete=False, encoding="utf-8") as f:
        f.write(cell)
        tmp_file = f.name
    try:
        env = os.environ.copy()
        env["LD_LIBRARY_PATH"] = "/content/Madar/lib:/content/Madar/bin:/content/Madar/extensions/ringtensor:/content/Madar/extensions/ringcjson:" + env.get("LD_LIBRARY_PATH", "")
        cwd = os.path.abspath("/content/Madar") if os.path.exists("/content/Madar") else os.path.abspath(".")
        proc = subprocess.Popen(
            [MADAR_BIN, tmp_file],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
            cwd=cwd
        )
        for out_line in proc.stdout:
            print(out_line, end="", flush=True)
        proc.wait()
    finally:
        if os.path.exists(tmp_file):
            os.remove(tmp_file)

ip = get_ipython()
ip.register_magic_function(madar, 'cell', 'ring')
print("✅ %%madar cell magic registered successfully!")

## 🔍 Step 6: Baseline Evaluation (Before Fine-Tuning)
Evaluate the pre-trained DeepSeek-R1-Distill-1.5B before fine-tuning with ChatML + `<think>` reasoning tags.

In [ ]:
%%madar
load "madar_ai.ring"

? "===================================================================="
? "=== Baseline Test: DeepSeek-R1-Distill-1.5B Base Model ==="
? "===================================================================="

cModelPath = "models/deepseek-r1-distill-qwen-1.5b-q4_0.gguf"
oModel = model_open(cModelPath)
oModel.setTemperature(0.6)

? "\n[Test 1: Arabic Prompt with Reasoning]"
cPrompt1 = oModel.formatChatPrompt("اكتب دالة بلغة رينج لعكس نص والتحقق مما إذا كان palindrome مع مثال تطبيقي.", "أنت مبرمج خبير ومتخصص في لغة البرمجة رينج (Ring).")
? oModel.generate(cPrompt1, 160)

? "\n[Test 2: English Prompt with Reasoning]"
cPrompt2 = oModel.formatChatPrompt("Write a Ring function to calculate the factorial of a number with sample usage.", "You are an expert Ring language programmer.")
? oModel.generate(cPrompt2, 160)

oModel.close()


## 🚀 Step 7: Launch Multi-Layer Native LoRA Training
Train across all **28 Transformer layers** using Madar's native Causal LM training engine.
- **Architecture**: 28 layers, dim 1536
- **LoRA Parameters**: Rank $r=16$, Alpha $\alpha=32.0$, Learning Rate $LR=0.0003$
- **Loss Function**: Exact Next-Token Cross-Entropy with high-precision decimal reporting
- **Hardware Acceleration**: Tesla T4 OpenCL GPU Zero-Copy LM Head

In [ ]:
%%madar
load "madar_ai.ring"
load "madar_trainer.ring"

decimals(6)

? "===================================================================="
? "  Madar AI - Multi-Layer LoRA Training (DeepSeek-R1-Distill-1.5B)     "
? "===================================================================="

cModelPath = "models/deepseek-r1-distill-qwen-1.5b-q4_0.gguf"
cDataPath  = "data/ring_instruct_bilingual.jsonl"

# 1. Load Base Model with GPU Acceleration
oModel = model_open(cModelPath)
oModel.setThreads(4)
oModel.setGPU(true)

# 2. Ingest Bilingual Dataset (1,460 Samples)
oDataset = new MadarInstructDataset(cDataPath, "chatml")
? f"Loaded Dataset: {oDataset.size()} Bilingual Ring Instruction Pairs"

# 3. Configure Multi-Layer LoRA Trainer across all 28 layers
# Rank: 16 | Alpha: 16.0 | Base LR: 0.00005 | Max Seq: 384
oTrainer = new MadarLoRATrainer(oModel, oDataset, 16, 16.0, 0.00005)
oTrainer.nEpochs       = 3
oTrainer.nMaxSeqLen    = 384
oTrainer.nLogInterval  = 10
oTrainer.nSaveInterval = 200
oTrainer.cOutputDir    = "adapters/deepseek_r1_lora"

# 4. Run Causal LM Training Loop
oTrainer.train(3)

# 5. Clean up
oModel.close()


## 📈 Step 8: Visualize Training Convergence
Plot the training loss curve across steps to inspect learning progress and convergence.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

csv_path = "adapters/deepseek_r1_lora/final/loss_history.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    plt.figure(figsize=(10, 5))
    plt.plot(df["step"], df["loss"], color="#2E86AB", linewidth=1.5, label="Training Loss")
    if len(df) > 20:
        df["rolling"] = df["loss"].rolling(window=20).mean()
        plt.plot(df["step"], df["rolling"], color="#D90429", linewidth=2.0, label="Smoothed (20 steps)")
    plt.title("DeepSeek-R1-Distill-1.5B Multi-Layer LoRA Convergence", fontsize=14, fontweight="bold")
    plt.xlabel("Step", fontsize=12)
    plt.ylabel("Cross-Entropy Loss", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Loss history file will be available once training finishes.")

## 💎 Step 9: Export Standalone Merged GGUF Model
Permanently fuse all 56 attention adapters across all 28 layers into a standalone `.gguf` file.

In [ ]:
%%madar
load "madar_ai.ring"

? "===================================================================="
? "  Madar AI - Direct GGUF Model Exporter (28-Layer Fusion)            "
? "===================================================================="

cBaseModel = "models/deepseek-r1-distill-qwen-1.5b-q4_0.gguf"
cMergedOut = "models/deepseek-r1-distill-1.5b-ring-bilingual-merged.gguf"
cAdapter   = "adapters/deepseek_r1_lora/final"

oModel = model_open(cBaseModel)
oModel.loadLoRAFolder(cAdapter)
oModel.exportMergedGGUF(cMergedOut)
oModel.close()

? "✅ Standalone Merged Model Exported: " + cMergedOut


## 🏆 Step 10: Final Evaluation of Merged Model
Verify that the fine-tuned model leverages its `<think>` reasoning and generates valid Ring code.

In [ ]:
%%madar
load "madar_ai.ring"

? "===================================================================="
? "  Testing Merged Model: DeepSeek-R1-Distill-1.5B (Ring Specialist)   "
? "===================================================================="

cMergedModel = "models/deepseek-r1-distill-1.5b-ring-bilingual-merged.gguf"
oModel = model_open(cMergedModel)
oModel.setTemperature(0.6)

? "\n==================== [Test 1: Arabic Prompt] ===================="
cPrompt1 = oModel.formatChatPrompt("اكتب دالة بلغة رينج لعكس نص والتحقق مما إذا كان palindrome مع مثال تطبيقي.", "أنت مبرمج خبير ومتخصص في لغة البرمجة رينج (Ring).")
? oModel.generate(cPrompt1, 200)

? "\n==================== [Test 2: English Prompt] ===================="
cPrompt2 = oModel.formatChatPrompt("Write a Ring function to calculate the factorial of a number with sample usage.", "You are an expert Ring language programmer.")
? oModel.generate(cPrompt2, 200)

oModel.close()


## 📦 Step 11: Package & Download Standalone Model and Adapters
Download the final merged `.gguf` file and the zip archive containing all 28-layer LoRA weights.

In [ ]:
from google.colab import files
import os

# 1. Compress Adapters
!zip -r deepseek_r1_lora.zip adapters/deepseek_r1_lora/final/
files.download("deepseek_r1_lora.zip")

# 2. Download Standalone GGUF Model
merged_file = "models/deepseek-r1-distill-1.5b-ring-bilingual-merged.gguf"
if os.path.exists(merged_file):
    print(f"⬇️ Downloading standalone GGUF model: {merged_file} ({os.path.getsize(merged_file)/(1024*1024):.1f} MB)...")
    files.download(merged_file)
